In [ ]:
# input
fasta_file = "../tmp/no_anno_homomer.fasta"
pred_file = "../tmp/pred_no_anno.tsv"
# output
output_with_metal_json_file = "./tmp/protenix_input_with_metal.json"

In [2]:
metal_type_str_to_input = {
"0": ("ion", "ZN"),
"1": ("ion", "CA"),
"2": ("ion", "MG"),
"3": ("ion", "MN"),
"4": ("ion", "FE"),
"5": ("ion", "CU"),
"6": ("ion", "NI"),
"7": ("ion", "CO"),
"8": ("ligand", "CCD_SF4"),
"9": ("ligand", "CCD_FES"),
"10": ("ligand", "CCD_F3S"),
}

In [3]:
import pandas as pd


df = pd.read_table(pred_file)
df

,seq_id,posi,pdb_seq_num,site,site_chain_name,site_max_chain_num,chain_num,rep,pfam_id,ted_id,anno_level,sp,org,sym,pred_seq_num,proba,metal_type,metal_group_type,avg_plddt,high_conf_pred_seq_num
0,Q8U3K7,35,36,"36,36,36","B,C,A",3,3,A0A172WEY9,NaN,NaN,NaN,0,pf,C3,"7,36,68","0.4167,0.2304,0.217","2,0,2","0,1,0",97.038,"7,36,68"
1,Q96T91,"120,123","121,124","121,121,124,124","B,A,A,B",2,2,Q96T91,NaN,NaN,NaN,1,hs,C2,"121,124","0.5492,0.2301","0,0","1,1",NaN,NaN
2,A8MYA2,"483,486","484,487","484,484,487,487","A,B,B,A",2,2,A8MYA2,NaN,NaN,NaN,1,hs,C2,"484,487","0.977,0.8893","0,0","1,1",NaN,NaN
3,O60296,"662,665","663,666","663,663,666,666","B,A,B,A",2,2,A0A2K5MI32,NaN,NaN,NaN,1,hs,C2,"663,666","0.7835,0.4423","1,0","1,1",NaN,NaN
4,Q49A92,"408,411","409,412","409,409,412,412","B,A,B,A",2,2,A0A6P7RDG8,NaN,NaN,NaN,1,hs,C2,"409,412,527","0.809,0.786,0.404","0,0,0","1,1,1",55.490,"409,412,527"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,Q8X8B1,"54,57,61","55,58,62","55,55,58,58,62,62","B,A,B,A,B,A",2,2,A0A826MMQ3,NaN,NaN,NaN,0,ec,C2,"55,58,62","0.9028,0.7848,0.3651","0,0,0","1,1,1",63.285,"55,58,62"
62,P38145,"53,102,107,192","54,103,108,193","54,103,108,193;103,103,108,193","A,D,A,A;D,A,A,A",2,4,A0A0P1KUL0,NaN,NaN,NaN,1,sc,C4,"54,103,108,193","0.3013,0.2544,0.8691,0.5507","0,0,0,2","1,1,1,1",96.085,"54,103,108,193"
63,P15408,"186,190","187,191","187,187,191,191","B,A,A,B",2,2,P51145,NaN,NaN,NaN,1,hs,C2,"141,187,191","0.2258,0.9186,0.6825","0,4,0","1,1,1",NaN,NaN
64,Q13336,331,295,"295,295,295","B,C,A",3,3,A0A1X9WEH5,NaN,NaN,NaN,1,hs,C3,332,0.2073,0,0,NaN,NaN


In [4]:
from Bio import SeqIO

id2seq = dict()
for r in SeqIO.parse(fasta_file, "fasta"):
    id2seq[r.id] = str(r.seq)

records = []
records_with_metal = []
for _, row in df.iterrows():
    seq = id2seq[row['seq_id']]

    metal2num = dict()
    for m in set(row['metal_type'].split(",")):
        if m in metal2num.keys():
            metal2num[m] += 1
        else:
            metal2num[m] = 1

    metal_dict_list = []
    for m, num in metal2num.items():
        ligand_type, metal_str = metal_type_str_to_input[m]
        if ligand_type == "ion":
            metal_dict_list.append({
                "ion": {
                    "ion": metal_str,
                    "count": num
                }
            })
        elif ligand_type == "ligand":
            metal_dict_list.append({
                "ligand": {
                    "ligand": f"{metal_str}",
                    "count": num
                }
            })
        else:
            raise ValueError

    pro_chain = [
        {
            "proteinChain": {
                "sequence": seq,
                "count": row['chain_num'],
                "msa": {
                    "precomputed_msa_dir": f"./tmp/msa_protenix/{row['seq_id']}/0",
                    "pairing_db": "uniref100",
                }
            },
        },
    ] 

    records.append({
        "name": row['seq_id'],
        "sequences": pro_chain
    })

    records_with_metal.append({
        "name": row['seq_id'],
        "sequences": pro_chain + metal_dict_list
    })

In [5]:
import json

with open(output_with_metal_json_file, "w", encoding="utf-8") as f:
    json.dump(records_with_metal, f, ensure_ascii=False, indent=4)